[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maierav/ai_oscp_neuro/blob/main/notebooks/ccf_penetration_figures.ipynb)

# OpenScope Predictive Processing — CCF Penetration Figures

Reproduce the 3D penetration renders and per-probe laminar cross-check figures for
any CCF-labeled ecephys session in **DANDI 001637**, and build the attachable CCF
sidecar tables.

**Runtime:** CPU is fine. Each session streams ~10–15 GB over HTTP via `remfile`
(no full download); a figure pair takes ~10–20 s. Open in Colab and run top to bottom.


In [ ]:
#@title Install dependencies
!pip -q install pynwb h5py remfile requests pandas numpy scipy matplotlib pyarrow
# brainglobe-atlasapi is optional (only needed for the translucent brain shell in 3D)
!pip -q install brainglobe-atlasapi 2>/dev/null || echo "brainglobe optional - skipping"


In [ ]:
#@title Get the package
import os
if not os.path.exists('openscope_ccf'):
    !git clone -q https://github.com/maierav/ai_oscp_neuro.git
    %cd ai_oscp_neuro
import sys; sys.path.insert(0,'.')
import openscope_ccf as o
print('openscope_ccf', o.__version__)


## 1. Pick a session
The registry lists all CCF-labeled sessions with subject, date, paradigm and DANDI asset id.

In [ ]:
idx = o.load_session_index()
idx

In [ ]:
#@title Choose one session
subject = 820459 #@param {type:"integer"}
row = idx[idx.subject==subject].iloc[0]
tag = f"{row.subject}_{row.date}"
print("session:", tag, "|", row.paradigm, "|", row.n_units, "units")

## 2. Build the CCF sidecars (attachable tables)
One per-unit and one per-channel Parquet, keyed to the NWB `units` / `electrodes` row indices.

In [ ]:
paths = o.build_session_sidecars(row.aid, str(row.subject), row.date, row.paradigm)
import pandas as pd
units = pd.read_parquet(paths['units'])
print("unit sidecar:", units.shape, "| QC units:", units.qc_pass.sum())
units.head()

### Attach CCF onto any analysis
Any result table that carries `unit_index` (SUA/MUA) or `electrode_row`/`channel`
(LFP/CSD) can be annotated in one call:

In [ ]:
import numpy as np
my_result = pd.DataFrame({'unit_index': np.arange(len(units)),
                          'response': np.random.rand(len(units))})
annotated = o.attach(my_result, tag, on='unit_index')
annotated[['unit_index','response','area','layer','group']].head()

## 3. Penetration figures
### 3D atlas context (best-fit tracks)

In [ ]:
pd_ = o.build_probe_data(row.aid)
mesh = None
try:
    mesh = o.load_root_mesh()   # translucent Allen brain shell (optional)
except Exception as e:
    print("brain shell unavailable, rendering without it:", str(e)[:80])
o.make_3d(pd_, tag, "fig_3d.png", brain_mesh=mesh)
from IPython.display import Image; Image("fig_3d.png")

### Laminar cross-check: CCF region/layer vs LFP power & MUA

In [ ]:
bg = None
try:
    from brainglobe_atlasapi import BrainGlobeAtlas
    bg = BrainGlobeAtlas("allen_mouse_25um")
except Exception:
    pass
o.make_laminar(pd_, tag, "fig_laminar.png", bg=bg)
Image("fig_laminar.png")

---
Built by the OpenScope PP CCF toolkit. Data: [DANDI 001637](https://dandiarchive.org/dandiset/001637).
CCF alignment: OpenScope community (discussion #163).